# Skin Disease Classification — v3

Multi-class skin disease classification using **DINOv2 ViT-B/14** (pre-trained on skin diseases).

**Datasets:**
- **SCIN** — 5,033 user-submitted phone photos from Google (HuggingFace)
- **SkinDisNet** — 1,710 smartphone photos from Bangladesh (Mendeley)
- **Skin Lesions** — Healthy skin images from ahmed-ai/skin-lesions-classification-dataset (HuggingFace)
- **DermNet** — Clinical atlas photos from Kaggle
- **SD-198** — Clinical images from HuggingFace

**Target Classes (6):**
Eczema, Fungal Infection, Acne, Psoriasis, Scabies, Healthy Skin

**Data Strategy:** Multi-source data pipeline combining SCIN, SkinDisNet, healthy skin,
DermNet, and SD-198 with confidence-based label filtering.

**Runtime:** Select GPU via *Runtime > Change runtime type > L4 GPU* (recommended)

**Storage:** Datasets are cached as zips on Google Drive. First run downloads/uploads normally
and saves to Drive. Subsequent runs restore from Drive in ~5-10 min instead of hours.

## Step 1: Setup

Uses Colab's local `/content/` disk for all data processing (fast I/O).
Google Drive is mounted early to **cache datasets as zips** — on subsequent runs,
data restores from Drive in ~5-10 min instead of re-downloading (5+ hours).

In [ ]:
# All data on Colab local disk (fast, ~100GB free)
import os

DATA_PATH = "/content/skin_disease_data"
MODEL_PATH = "/content/skin_disease_models"
os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

print(f"Data:   {DATA_PATH}")
print(f"Models: {MODEL_PATH}")

In [ ]:
# ── Mount Google Drive & restore cached datasets ─────────────────────────────
import shutil, zipfile
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = "/content/drive/MyDrive/skin_disease_project/data"
os.makedirs(DRIVE_DATA, exist_ok=True)

# Fast-restore: copy zips from Drive → /content/ and unzip
RESTORE_MAP = {
    "scin.zip":              f"{DATA_PATH}/scin",
    "scin_images.zip":       f"{DATA_PATH}/scin_images",
    "skindisnet.zip":        f"{DATA_PATH}/skindisnet",
    "healthy_raw.zip":       f"{DATA_PATH}/healthy_raw",
    "healthy_images.zip":    f"{DATA_PATH}/healthy_images",
    "dermnet.zip":           f"{DATA_PATH}/dermnet",
    "dermnet_targeted.zip":  f"{DATA_PATH}/dermnet_targeted",
    "sd198_targeted.zip":    f"{DATA_PATH}/sd198_targeted",
    "combined_v4.zip":       f"{DATA_PATH}/combined",
}

restored, needed = [], []
for zip_name, target_dir in RESTORE_MAP.items():
    zip_on_drive = f"{DRIVE_DATA}/{zip_name}"
    if os.path.exists(target_dir) and len(os.listdir(target_dir)) > 0:
        print(f"  {zip_name}: already on local disk")
        restored.append(zip_name)
    elif os.path.exists(zip_on_drive):
        if not zipfile.is_zipfile(zip_on_drive):
            print(f"  {zip_name}: corrupt zip on Drive, will re-download")
            os.remove(zip_on_drive)
            needed.append(zip_name)
            continue
        print(f"  {zip_name}: restoring from Drive...", end=" ", flush=True)
        shutil.copy2(zip_on_drive, f"/tmp/{zip_name}")
        ret = os.system(f'cd {DATA_PATH} && unzip -q -o /tmp/{zip_name}')
        os.remove(f"/tmp/{zip_name}")
        if ret != 0:
            print(f"FAILED (unzip exit code {ret})")
            needed.append(zip_name)
            continue
        print("done")
        restored.append(zip_name)
    else:
        needed.append(zip_name)

if restored:
    print(f"\nRestored from Drive: {', '.join(restored)}")
if needed:
    print(f"Not on Drive yet (will save after first download): {', '.join(needed)}")
else:
    print("\nAll datasets restored from Drive — download cells will use local cache.")

In [ ]:
# Install and import everything
!pip install -q datasets transformers

import io, json, random, shutil, ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, datasets
from transformers import AutoModel
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Handle truncated images gracefully during training
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Auto-detect GPU and set optimal batch size
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name or 'H100' in gpu_name:
        BATCH_SIZE = 64
        ACCUM_STEPS = 1
    elif 'L4' in gpu_name:
        BATCH_SIZE = 16
        ACCUM_STEPS = 2
    else:  # T4 or unknown
        BATCH_SIZE = 8
        ACCUM_STEPS = 4
else:
    gpu_name = "CPU"
    BATCH_SIZE = 4
    ACCUM_STEPS = 8

print(f"PyTorch: {torch.__version__} | Device: {device}")
print(f"GPU: {gpu_name} | Batch: {BATCH_SIZE} | Accum: {ACCUM_STEPS} | Effective: {BATCH_SIZE * ACCUM_STEPS}")

## Step 2: Label Mapping

SCIN labels are **strings** (e.g. `"Eczema"`, `"Acne vulgaris"`).
The `weighted_skin_condition_label` column is a **JSON string** of a dict — must be parsed.
There's also `dermatologist_skin_condition_on_label_name` which is a plain string.

SkinDisNet folders use **abbreviations**: AD, CD, EC, SD, SC, TC.

In [ ]:
# ── Classes and label mapping ──────────────────────────────────────────────────
TARGET_CLASSES = ['acne', 'eczema', 'fungal', 'healthy', 'psoriasis', 'scabies']
NUM_CLASSES = len(TARGET_CLASSES)

LABEL_MAPPING = {
    # Psoriasis
    'psoriasis': 'psoriasis',
    'plaque psoriasis': 'psoriasis',
    'guttate psoriasis': 'psoriasis',
    'psoriasis vulgaris': 'psoriasis',
    # Scabies
    'scabies': 'scabies',
    # Eczema
    'eczema': 'eczema',
    'atopic dermatitis': 'eczema',
    'contact dermatitis': 'eczema',
    'allergic contact dermatitis': 'eczema',
    'irritant contact dermatitis': 'eczema',
    'seborrheic dermatitis': 'eczema',
    'dermatitis': 'eczema',
    'eczematous dermatitis': 'eczema',
    'nummular eczema': 'eczema',
    'acute and chronic dermatitis': 'eczema',
    'nummular dermatitis': 'eczema',
    'perioral dermatitis': 'eczema',
    'stasis dermatitis': 'eczema',
    # Fungal
    'fungal': 'fungal',
    'tinea': 'fungal',
    'tinea corporis': 'fungal',
    'tinea pedis': 'fungal',
    'tinea versicolor': 'fungal',
    'tinea capitis': 'fungal',
    'tinea cruris': 'fungal',
    'tinea manum': 'fungal',
    'tinea faciale': 'fungal',
    'ringworm': 'fungal',
    'candidiasis': 'fungal',
    'onychomycosis': 'fungal',
    # Acne
    'acne': 'acne',
    'acne vulgaris': 'acne',
    'acne keloidalis nuchae': 'acne',
    'folliculitis': 'acne',
    # Healthy
    'healthy': 'healthy',
    'normal': 'healthy',
}

SCIN_CONFIDENCE_THRESHOLD = 0.5

def map_label(raw):
    """Map a raw label string to one of TARGET_CLASSES, or None."""
    if raw is None:
        return None
    label = str(raw).lower().strip()
    if label in LABEL_MAPPING:
        return LABEL_MAPPING[label]
    for key, target in LABEL_MAPPING.items():
        if len(key) >= 4 and key in label:
            return target
    return None

def parse_weighted_label(raw_str):
    """Parse SCIN's weighted_skin_condition_label (JSON/dict string) to best mapped class.
    Skips labels below SCIN_CONFIDENCE_THRESHOLD."""
    if not raw_str or pd.isna(raw_str):
        return None

    if isinstance(raw_str, dict):
        label_dict = raw_str
    elif isinstance(raw_str, str):
        if raw_str.strip() == '{}':
            return None
        try:
            label_dict = json.loads(raw_str)
        except (json.JSONDecodeError, ValueError):
            try:
                label_dict = ast.literal_eval(raw_str)
            except (ValueError, SyntaxError):
                return map_label(raw_str)
    else:
        return map_label(raw_str)

    if not isinstance(label_dict, dict):
        return map_label(str(label_dict))

    try:
        sorted_keys = sorted(label_dict, key=lambda k: float(label_dict[k]), reverse=True)
    except (ValueError, TypeError):
        sorted_keys = list(label_dict.keys())

    # Skip if highest-confidence label is below threshold
    if sorted_keys:
        try:
            if float(label_dict[sorted_keys[0]]) < SCIN_CONFIDENCE_THRESHOLD:
                return None
        except (ValueError, TypeError):
            pass

    for condition in sorted_keys:
        mapped = map_label(condition)
        if mapped:
            return mapped
    return None

assert map_label('Eczema') == 'eczema'
assert map_label('Acne vulgaris') == 'acne'
assert map_label('Acne keloidalis nuchae') == 'acne'
assert map_label('Psoriasis') == 'psoriasis'
assert map_label('Plaque psoriasis') == 'psoriasis'
assert map_label('Scabies') == 'scabies'
assert map_label('Guttate psoriasis') == 'psoriasis'
assert map_label('Onychomycosis') == 'fungal'
assert map_label('Nummular dermatitis') == 'eczema'
assert parse_weighted_label("{'Eczema': 0.67, 'Psoriasis': 0.11}") == 'eczema'
assert parse_weighted_label('{"Tinea": 0.5, "Acne": 0.3}') == 'fungal'
assert parse_weighted_label('{"Eczema": 0.3}') is None  # below threshold
assert parse_weighted_label(None) is None
assert parse_weighted_label('{}') is None
print(f"Target classes: {TARGET_CLASSES}")
print(f"SCIN confidence threshold: {SCIN_CONFIDENCE_THRESHOLD}")
print("Label mapping tests passed")

## Step 3: Download & Process SCIN Dataset

SCIN stores images in `image_1_path` (HuggingFace Image type → decoded as PIL).
Labels come from `dermatologist_skin_condition_on_label_name` (plain string)
with fallback to `weighted_skin_condition_label` (JSON-encoded dict string).

In [ ]:
from datasets import load_dataset, load_from_disk

SCIN_PATH = f"{DATA_PATH}/scin"

if os.path.exists(SCIN_PATH):
    print("Loading SCIN from local cache...")
    scin_dataset = load_from_disk(SCIN_PATH)
else:
    print("Downloading SCIN from HuggingFace (first time, ~12GB)...")
    scin_dataset = load_dataset("google/scin", split="train")
    scin_dataset.save_to_disk(SCIN_PATH)
    print("Saved to local disk")

# NOTE: scin.zip Drive save disabled — too large (~11GB), wastes Drive space.
# The processed scin_images.zip and combined_v3.zip are sufficient for restore.
# if not os.path.exists(f"{DRIVE_DATA}/scin.zip"):
#     print("Saving SCIN to Drive (one-time, may take a while)...")
#     !cd {DATA_PATH} && zip -r -q "{DRIVE_DATA}/scin.zip.tmp" scin/
#     os.rename(f"{DRIVE_DATA}/scin.zip.tmp", f"{DRIVE_DATA}/scin.zip")
#     print("Saved to Drive")

print(f"SCIN loaded: {len(scin_dataset)} samples")

In [ ]:
# ── Debug: inspect SCIN sample to verify field types ─────────────────────────
sample = scin_dataset[0]
print("=== Sample 0 key fields ===")
for key in ['dermatologist_skin_condition_on_label_name',
            'weighted_skin_condition_label',
            'image_1_path', 'image_2_path', 'image_3_path']:
    val = sample.get(key, 'MISSING')
    print(f"  {key}: {type(val).__name__} = {repr(val)[:200]}")

# Use column-level access (fast, no image decoding)
columns = scin_dataset.column_names
print(f"\nAll columns: {columns}")
has_direct = sum(1 for v in scin_dataset['dermatologist_skin_condition_on_label_name'] if v)
has_weighted = sum(1 for v in scin_dataset['weighted_skin_condition_label'] if v)
print(f"Samples with direct label: {has_direct}/{len(scin_dataset)}")
print(f"Samples with weighted label: {has_weighted}/{len(scin_dataset)}")

In [ ]:
# ── Process SCIN images ───────────────────────────────────────────────────────
SCIN_IMAGES_PATH = f"{DATA_PATH}/scin_images"
os.makedirs(SCIN_IMAGES_PATH, exist_ok=True)
scin_data = []

# Check for existing processed images
existing_files = list(Path(SCIN_IMAGES_PATH).glob("*.jpg"))
existing_labels = Counter(f.stem.rsplit('_', 1)[0] for f in existing_files)
non_empty = sum(1 for c in TARGET_CLASSES if existing_labels.get(c, 0) > 0)

if len(existing_files) > 100 and non_empty >= 3:
    print(f"SCIN already processed: {len(existing_files)} images, {non_empty} classes")
    for img_path in existing_files:
        label = img_path.stem.rsplit('_', 1)[0]
        if label in TARGET_CLASSES:
            scin_data.append({'path': str(img_path), 'label': label, 'source': 'scin'})
else:
    if existing_files:
        print("Clearing stale SCIN images...")
        shutil.rmtree(SCIN_IMAGES_PATH)
        os.makedirs(SCIN_IMAGES_PATH, exist_ok=True)

    print("Processing SCIN images...")
    save_failures = 0
    unmapped_labels = Counter()

    for idx, sample in enumerate(tqdm(scin_dataset, desc="SCIN")):
        mapped = None
        direct = sample.get('dermatologist_skin_condition_on_label_name', None)
        if direct:
            if isinstance(direct, str) and direct.startswith('['):
                try:
                    labels_list = ast.literal_eval(direct)
                    if isinstance(labels_list, list) and labels_list:
                        direct = labels_list[0]
                except (ValueError, SyntaxError):
                    pass
            mapped = map_label(direct)
        if not mapped:
            weighted = sample.get('weighted_skin_condition_label', None)
            if weighted:
                mapped = parse_weighted_label(weighted)

        if not mapped:
            if direct:
                unmapped_labels[direct] += 1
            continue

        img = None
        for img_col in ['image_1_path', 'image_2_path', 'image_3_path']:
            img_val = sample.get(img_col, None)
            if img_val is not None:
                if isinstance(img_val, Image.Image):
                    img = img_val
                    break
                elif isinstance(img_val, dict) and 'bytes' in img_val:
                    img = Image.open(io.BytesIO(img_val['bytes']))
                    break
                elif isinstance(img_val, dict) and 'path' in img_val:
                    try:
                        img = Image.open(img_val['path'])
                        break
                    except Exception:
                        continue

        if img is None:
            save_failures += 1
            continue

        img_path = f"{SCIN_IMAGES_PATH}/{mapped}_{idx}.jpg"
        try:
            img.convert('RGB').save(img_path)
            scin_data.append({'path': img_path, 'label': mapped, 'source': 'scin'})
        except Exception:
            save_failures += 1

    if save_failures:
        print(f"Warning: {save_failures} images failed to save/load")
    if unmapped_labels:
        print(f"Top unmapped labels: {unmapped_labels.most_common(10)}")

# Save processed images to Drive (runs on both code paths — idempotent check)
if not os.path.exists(f"{DRIVE_DATA}/scin_images.zip"):
    print("Saving processed SCIN images to Drive (one-time)...")
    !cd {DATA_PATH} && zip -r -q "{DRIVE_DATA}/scin_images.zip.tmp" scin_images/
    os.rename(f"{DRIVE_DATA}/scin_images.zip.tmp", f"{DRIVE_DATA}/scin_images.zip")
    print("Saved to Drive")

print(f"\nSCIN: {len(scin_data)} images mapped")
scin_dist = Counter(d['label'] for d in scin_data)
for cls in TARGET_CLASSES:
    print(f"  {cls}: {scin_dist.get(cls, 0)}")

## Step 4: Upload & Process SkinDisNet Dataset

Download from: https://data.mendeley.com/datasets/yj3md44hxg/2

Upload the zip when prompted. Structure: `Augmented/` and `Preprocessed/` with subfolders AD, CD, EC, SD, SC, TC.

In [ ]:
SKINDISNET_PATH = f"{DATA_PATH}/skindisnet"

if os.path.exists(SKINDISNET_PATH) and len(os.listdir(SKINDISNET_PATH)) > 0:
    print("SkinDisNet already extracted")
else:
    print("Upload SkinDisNet.zip (download from Mendeley link above):")
    from google.colab import files
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    print(f"Extracting {zip_name}...")
    os.makedirs(SKINDISNET_PATH, exist_ok=True)
    with zipfile.ZipFile(f"/content/{zip_name}", 'r') as zf:
        zf.extractall(SKINDISNET_PATH)
    os.remove(f"/content/{zip_name}")
    print("Extracted")

# Save to Drive for fast restore on future sessions (zip directly to Drive)
if not os.path.exists(f"{DRIVE_DATA}/skindisnet.zip"):
    print("Saving SkinDisNet to Drive (one-time)...")
    !cd {DATA_PATH} && zip -r -q "{DRIVE_DATA}/skindisnet.zip.tmp" skindisnet/
    os.rename(f"{DRIVE_DATA}/skindisnet.zip.tmp", f"{DRIVE_DATA}/skindisnet.zip")
    print("Saved to Drive")

# Show structure
print("\nSkinDisNet structure:")
for root, dirs, files_list in os.walk(SKINDISNET_PATH):
    depth = root.replace(SKINDISNET_PATH, '').count(os.sep)
    indent = '  ' * depth
    name = os.path.basename(root)
    n_imgs = len([f for f in files_list if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if n_imgs > 0 or dirs:
        print(f"{indent}{name}/ ({n_imgs} images)")

In [ ]:
# ── Process SkinDisNet (Preprocessed only — skip Augmented to prevent leakage) ─
# Note: SKINDISNET_MAPPING still maps to scabies/psoriasis. These images are
# processed here but filtered out downstream by valid_classes (from TARGET_CLASSES).
skindisnet_data = []

SKINDISNET_MAPPING = {
    # Full names
    'atopic dermatitis': 'eczema', 'contact dermatitis': 'eczema',
    'eczema': 'eczema', 'seborrheic dermatitis': 'eczema',
    'scabies': 'scabies', 'tinea corporis': 'fungal',
    'acne': 'acne', 'psoriasis': 'psoriasis',
    'healthy': 'healthy', 'normal': 'healthy',
    # Abbreviations used in SkinDisNet folder names
    'ad': 'eczema', 'cd': 'eczema', 'ec': 'eczema',
    'sd': 'eczema', 'sc': 'scabies', 'tc': 'fungal',
}

# Only walk Preprocessed/ to avoid data leakage from Augmented/ variants
preprocessed_root = os.path.join(SKINDISNET_PATH, 'Preprocessed')
if not os.path.exists(preprocessed_root):
    # Try case-insensitive search
    for entry in os.listdir(SKINDISNET_PATH):
        if entry.lower() == 'preprocessed':
            preprocessed_root = os.path.join(SKINDISNET_PATH, entry)
            break

if not os.path.exists(preprocessed_root):
    print(f"WARNING: Preprocessed folder not found in {SKINDISNET_PATH}")
    print("Available folders:", os.listdir(SKINDISNET_PATH))
else:
    for root, dirs, files_list in os.walk(preprocessed_root):
        folder_name = os.path.basename(root).lower()
        mapped = SKINDISNET_MAPPING.get(folder_name)
        if mapped:
            for f in files_list:
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    skindisnet_data.append({
                        'path': os.path.join(root, f),
                        'label': mapped,
                        'source': 'skindisnet'
                    })

print(f"SkinDisNet (Preprocessed only): {len(skindisnet_data)} images mapped")
skindisnet_dist = Counter(d['label'] for d in skindisnet_data)
for cls in TARGET_CLASSES:
    print(f"  {cls}: {skindisnet_dist.get(cls, 0)}")

## Step 4b: Download & Process Healthy Skin (ahmed-ai)

Neither SCIN nor SkinDisNet contain healthy skin samples. We source them from
`ahmed-ai/skin-lesions-classification-dataset` on HuggingFace, which has a labeled
"Healthy" class (label index 7). Only the Healthy class is extracted — other disease
classes are already covered by SCIN/SkinDisNet.

In [ ]:
# ── Download & filter Healthy class from ahmed-ai dataset ─────────────────
from datasets import concatenate_datasets

HEALTHY_RAW_PATH = f"{DATA_PATH}/healthy_raw"

if os.path.exists(HEALTHY_RAW_PATH):
    print("Loading healthy dataset from local cache...")
    healthy_dataset = load_from_disk(HEALTHY_RAW_PATH)
else:
    print("Downloading ahmed-ai/skin-lesions-classification-dataset from HuggingFace...")
    ds = load_dataset("ahmed-ai/skin-lesions-classification-dataset")

    # Resolve Healthy label index dynamically (don't hardcode)
    label_names = ds['train'].features['label'].names
    healthy_idx = label_names.index('Healthy')
    print(f"  Healthy class index: {healthy_idx} (from {label_names})")

    # Filter to Healthy class only across all splits
    healthy_splits = []
    for split_name, split in ds.items():
        healthy_split = split.filter(lambda x: x['label'] == healthy_idx)
        print(f"  {split_name}: {len(healthy_split)} healthy images (of {len(split)} total)")
        if len(healthy_split) > 0:
            healthy_splits.append(healthy_split)

    if not healthy_splits:
        raise ValueError("No Healthy images found in any split of ahmed-ai dataset")

    healthy_dataset = concatenate_datasets(healthy_splits)
    healthy_dataset.save_to_disk(HEALTHY_RAW_PATH)
    print("Saved filtered dataset to local disk")

# Save to Drive for fast restore
if not os.path.exists(f"{DRIVE_DATA}/healthy_raw.zip"):
    print("Saving healthy dataset to Drive (one-time)...")
    !cd {DATA_PATH} && zip -r -q "{DRIVE_DATA}/healthy_raw.zip.tmp" healthy_raw/
    os.rename(f"{DRIVE_DATA}/healthy_raw.zip.tmp", f"{DRIVE_DATA}/healthy_raw.zip")
    print("Saved to Drive")

print(f"\nHealthy dataset: {len(healthy_dataset)} images")

In [ ]:
# ── Process healthy skin images ───────────────────────────────────────────
HEALTHY_IMAGES_PATH = f"{DATA_PATH}/healthy_images"
os.makedirs(HEALTHY_IMAGES_PATH, exist_ok=True)
healthy_data = []

existing_files = list(Path(HEALTHY_IMAGES_PATH).glob("*.jpg"))

if len(existing_files) > 10:
    print(f"Healthy images already processed: {len(existing_files)} images")
    for img_path in existing_files:
        healthy_data.append({'path': str(img_path), 'label': 'healthy', 'source': 'healthy_skin'})
else:
    if existing_files:
        print("Clearing stale healthy images...")
        shutil.rmtree(HEALTHY_IMAGES_PATH)
        os.makedirs(HEALTHY_IMAGES_PATH, exist_ok=True)

    print("Processing healthy skin images...")
    save_failures = 0

    for idx, sample in enumerate(tqdm(healthy_dataset, desc="Healthy")):
        img = sample.get('image')

        if not isinstance(img, Image.Image):
            save_failures += 1
            continue

        img_path = f"{HEALTHY_IMAGES_PATH}/healthy_{idx}.jpg"
        try:
            img.convert('RGB').save(img_path)
            healthy_data.append({'path': img_path, 'label': 'healthy', 'source': 'healthy_skin'})
        except Exception:
            save_failures += 1

    if save_failures:
        print(f"Warning: {save_failures} images failed to save/load")

# Save processed images to Drive (idempotent)
if not os.path.exists(f"{DRIVE_DATA}/healthy_images.zip"):
    print("Saving processed healthy images to Drive (one-time)...")
    !cd {DATA_PATH} && zip -r -q "{DRIVE_DATA}/healthy_images.zip.tmp" healthy_images/
    os.rename(f"{DRIVE_DATA}/healthy_images.zip.tmp", f"{DRIVE_DATA}/healthy_images.zip")
    print("Saved to Drive")

print(f"\nHealthy skin: {len(healthy_data)} images")

## Step 4c: DermNet (Kaggle)

**DermNet** (~19,500 images, 23 classes) provides clinical atlas photos.
To re-enable: populate `DERMNET_FOLDER_MAP` with target folders (e.g., Psoriasis, Acne).
Requires Kaggle API credentials — set `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab Secrets.

In [ ]:
# ── DermNet ────────────────────────────────────────────────────────────────────
DERMNET_PATH = f"{DATA_PATH}/dermnet"
DERMNET_IMAGES_PATH = f"{DATA_PATH}/dermnet_targeted"
os.makedirs(DERMNET_IMAGES_PATH, exist_ok=True)
dermnet_data = []

# To enable: populate DERMNET_FOLDER_MAP and uncomment the processing loop below.
# DERMNET_FOLDER_MAP = {
#     'Acne and Rosacea Photos':  'acne',
#     'Psoriasis':                'psoriasis',
#     'Scabies Lyme Disease and other Infestations': 'scabies',
# }
DERMNET_FOLDER_MAP = {}

existing_files = list(Path(DERMNET_IMAGES_PATH).glob("*.jpg"))

if len(existing_files) > 50:
    print(f"DermNet (targeted) already processed: {len(existing_files)} images")
    for img_path in existing_files:
        label = img_path.stem.rsplit('_', 1)[0]
        if label in TARGET_CLASSES:
            dermnet_data.append({'path': str(img_path), 'label': label, 'source': 'dermnet'})
elif not DERMNET_FOLDER_MAP:
    print("DermNet: skipped (DERMNET_FOLDER_MAP is empty)")

# Save processed images to Drive (idempotent)
if dermnet_data and not os.path.exists(f"{DRIVE_DATA}/dermnet_targeted.zip"):
    print("Saving processed DermNet images to Drive (one-time)...")
    shutil.make_archive("/tmp/dermnet_targeted", 'zip', DATA_PATH, 'dermnet_targeted')
    shutil.move("/tmp/dermnet_targeted.zip", f"{DRIVE_DATA}/dermnet_targeted.zip")
    print("Saved to Drive")

print(f"\nDermNet (targeted): {len(dermnet_data)} images mapped")
dermnet_dist = Counter(d['label'] for d in dermnet_data)
for cls in TARGET_CLASSES:
    print(f"  {cls}: {dermnet_dist.get(cls, 0)}")

## Step 4d: SD-198 (HuggingFace)

**SD-198** (6,584 images, 198 fine-grained classes) provides additional clinical images.
To re-enable: populate `SD198_TARGET_CLASSES` with desired classes.
This is an unofficial HuggingFace upload — the cell handles `DatasetNotFoundError` gracefully.

In [ ]:
# ── SD-198 ────────────────────────────────────────────────────────────────────
SD198_IMAGES_PATH = f"{DATA_PATH}/sd198_targeted"
os.makedirs(SD198_IMAGES_PATH, exist_ok=True)
sd198_data = []

# To enable: populate SD198_TARGET_CLASSES and uncomment the processing loop below.
SD198_TARGET_CLASSES = set()

existing_files = list(Path(SD198_IMAGES_PATH).glob("*.jpg"))

if len(existing_files) > 10:
    print(f"SD-198 (targeted) already processed: {len(existing_files)} images")
    for img_path in existing_files:
        label = img_path.stem.rsplit('_', 1)[0]
        if label in TARGET_CLASSES:
            sd198_data.append({'path': str(img_path), 'label': label, 'source': 'sd198'})
elif not SD198_TARGET_CLASSES:
    print("SD-198: skipped (SD198_TARGET_CLASSES is empty)")

# Save processed images to Drive (idempotent)
if sd198_data and not os.path.exists(f"{DRIVE_DATA}/sd198_targeted.zip"):
    print("Saving processed SD-198 images to Drive (one-time)...")
    shutil.make_archive("/tmp/sd198_targeted", 'zip', DATA_PATH, 'sd198_targeted')
    shutil.move("/tmp/sd198_targeted.zip", f"{DRIVE_DATA}/sd198_targeted.zip")
    print("Saved to Drive")

print(f"\nSD-198 (targeted): {len(sd198_data)} images mapped")
sd198_dist = Counter(d['label'] for d in sd198_data)
for cls in TARGET_CLASSES:
    print(f"  {cls}: {sd198_dist.get(cls, 0)}")

## Step 5: Combine, Filter & Split

In [ ]:
# ── Combine datasets ──────────────────────────────────────────────────────────
# Original sources (used for train/val/test splits)
original_data = scin_data + skindisnet_data + healthy_data
# New sources (train only -- keeps test set fair for baseline comparison)
new_data = dermnet_data + sd198_data
all_data = original_data + new_data

print(f"Total: {len(all_data)} images ({len(original_data)} original + {len(new_data)} new)")

source_dist = Counter(d['source'] for d in all_data)
print(f"\nBy source: {dict(source_dist)}")

print("\nBy class AND source:")
for cls in TARGET_CLASSES:
    cls_items = [d for d in all_data if d['label'] == cls]
    src_counts = Counter(d['source'] for d in cls_items)
    parts = ", ".join(f"{s}: {c}" for s, c in sorted(src_counts.items()))
    print(f"  {cls}: {len(cls_items)} ({parts})")

missing = [c for c in TARGET_CLASSES if not any(d['label'] == c for d in all_data)]
if missing:
    print(f"\nWARNING: No images for: {missing}")
    print("These classes will be empty. Check label mappings above.")

In [ ]:
# ── Filter: keep only classes with enough images for splitting ─────────────────
MIN_CLASS_IMAGES = 10
orig_class_dist = Counter(d['label'] for d in original_data)
valid_classes = [c for c in TARGET_CLASSES if orig_class_dist.get(c, 0) >= MIN_CLASS_IMAGES]
excluded = [f"{c} ({orig_class_dist.get(c, 0)})" for c in TARGET_CLASSES if c not in valid_classes]
if excluded:
    print(f"Excluded classes (<{MIN_CLASS_IMAGES} original images): {', '.join(excluded)}")
filtered_original = [d for d in original_data if d['label'] in valid_classes]
filtered_new = [d for d in new_data if d['label'] in valid_classes]

print(f"Using {len(valid_classes)} classes: {valid_classes}")
print(f"Original images: {len(filtered_original)} | New images: {len(filtered_new)}")

# ── Split ORIGINAL data 80/10/10 (preserves test set for fair comparison) ─────
df_orig = pd.DataFrame(filtered_original)
df_orig['_strat_key'] = df_orig['label'] + '_' + df_orig['source']

def pick_strat_col(frame, min_count):
    """Use compound (label+source) stratification if all groups are large enough."""
    counts = frame['_strat_key'].value_counts()
    if counts.min() >= min_count:
        print(f"Stratifying by (label, source) -- smallest group has {counts.min()} samples")
        return '_strat_key'
    small = counts[counts < min_count]
    print(f"Falling back to label-only stratification (groups too small: {dict(small)})")
    return 'label'

strat_col = pick_strat_col(df_orig, min_count=3)
train_orig, temp_df = train_test_split(df_orig, test_size=0.2, stratify=df_orig[strat_col], random_state=42)

temp_strat_col = pick_strat_col(temp_df, min_count=2) if strat_col == '_strat_key' else 'label'
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df[temp_strat_col], random_state=42)

# New data goes to train ONLY (no val/test contamination)
df_new = pd.DataFrame(filtered_new)
train_df = pd.concat([train_orig, df_new], ignore_index=True)

print(f"\nTrain: {len(train_df)} ({len(train_orig)} original + {len(df_new)} new)")
print(f"Val: {len(val_df)} (original only) | Test: {len(test_df)} (original only)")

for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    src_pct = split_df['source'].value_counts(normalize=True)
    parts = ", ".join(f"{s}: {p*100:.0f}%" for s, p in src_pct.items())
    print(f"  {split_name} source mix: {parts}")

In [ ]:
# ── Organize into ImageFolder structure ────────────────────────────────────────
COMBINED_DIR = Path(f"{DATA_PATH}/combined")

VERSION_MARKER = COMBINED_DIR / '.data_version'
CURRENT_VERSION = 'v4_class_swap'

existing_classes = (
    {d.name for d in (COMBINED_DIR / 'train').iterdir() if d.is_dir()}
    if (COMBINED_DIR / 'train').exists() else set()
)

needs_rebuild = (
    not (COMBINED_DIR / 'train').exists()
    or existing_classes != set(valid_classes)
    or not (VERSION_MARKER.exists() and VERSION_MARKER.read_text().strip() == CURRENT_VERSION)
    or len(list((COMBINED_DIR / 'train').glob('*/*'))) < 100
)

if not needs_rebuild:
    print(f"Dataset already organized (version={CURRENT_VERSION}): {existing_classes}")
else:
    if COMBINED_DIR.exists():
        shutil.rmtree(COMBINED_DIR)

    print("Organizing into folders...")
    for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        for cls in valid_classes:
            (COMBINED_DIR / split_name / cls).mkdir(parents=True, exist_ok=True)

        for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=split_name):
            src = Path(row['path'])
            if src.exists():
                dst = COMBINED_DIR / split_name / row['label'] / f"{row['source']}_{src.name}"
                if not dst.exists():
                    shutil.copy2(src, dst)

    for split in ['train', 'val', 'test']:
        for cls_dir in (COMBINED_DIR / split).iterdir():
            if cls_dir.is_dir() and not any(cls_dir.iterdir()):
                cls_dir.rmdir()
                print(f"  Removed empty dir: {split}/{cls_dir.name}")

    VERSION_MARKER.write_text(CURRENT_VERSION)

    # Delete old combined zips from Drive
    for old_zip_name in ['combined.zip', 'combined_v2.zip', 'combined_v3.zip']:
        old_zip = f"{DRIVE_DATA}/{old_zip_name}"
        if os.path.exists(old_zip):
            os.remove(old_zip)
            print(f"Deleted old {old_zip_name} from Drive")

    # Cache as combined_v4.zip
    combined_zip = f"{DRIVE_DATA}/combined_v4.zip"
    if not os.path.exists(combined_zip):
        print("Caching combined/ to Drive as combined_v4.zip...", end=" ", flush=True)
        shutil.make_archive(f"/tmp/combined", 'zip', DATA_PATH, 'combined')
        shutil.move("/tmp/combined.zip", combined_zip)
        print(f"done ({os.path.getsize(combined_zip)/1e6:.0f} MB)")

# Verify with per-class-per-split counts
splits = ['train', 'val', 'test']
print("\nDataset summary (per class per split):")
print(f"{'Class':<15} {'Train':>7} {'Val':>7} {'Test':>7} {'Total':>7}")
print("-" * 50)
totals = {s: 0 for s in splits}
for cls in valid_classes:
    counts = {s: len(list((COMBINED_DIR / s / cls).glob('*'))) if (COMBINED_DIR / s / cls).exists() else 0 for s in splits}
    for s in splits:
        totals[s] += counts[s]
    total = sum(counts.values())
    print(f"{cls:<15} {counts['train']:>7} {counts['val']:>7} {counts['test']:>7} {total:>7}")
print("-" * 50)
print(f"{'TOTAL':<15} {totals['train']:>7} {totals['val']:>7} {totals['test']:>7} {sum(totals.values()):>7}")

In [ ]:
# ── Data Expansion Summary ────────────────────────────────────────────────────
print("=" * 70)
print("DATA EXPANSION SUMMARY")
print("=" * 70)

# Per-class breakdown by source
sources = ['scin', 'skindisnet', 'healthy_skin', 'dermnet', 'sd198']
source_totals = Counter()
header = f"{'Class':<12}" + "".join(f"{s:>12}" for s in sources) + f"{'TOTAL':>10}"
print(header)
print("-" * len(header))

for cls in valid_classes:
    cls_items = [d for d in all_data if d['label'] == cls]
    src_counts = Counter(d['source'] for d in cls_items)
    source_totals += src_counts
    row = f"{cls:<12}"
    for s in sources:
        count = src_counts.get(s, 0)
        row += f"{count:>12}" if count > 0 else f"{'—':>12}"
    row += f"{len(cls_items):>10}"
    print(row)

print("-" * len(header))
row = f"{'TOTAL':<12}"
for s in sources:
    count = source_totals.get(s, 0)
    row += f"{count:>12}" if count > 0 else f"{'—':>12}"
row += f"{len(all_data):>10}"
print(row)

# Use filesystem counts (works on both fresh-build and cache-restore paths)
print(f"\nSplit counts (from filesystem):")
for split in ['train', 'val', 'test']:
    split_dir = COMBINED_DIR / split
    count = len(list(split_dir.glob('*/*'))) if split_dir.exists() else 0
    print(f"  {split}: {count} images")

print(f"\nNew data goes to TRAIN ONLY — val/test are unchanged for fair comparison.")
print("=" * 70)

## Step 6: Data Loading & Augmentation

In [ ]:
IMG_SIZE = 224
NUM_WORKERS = 2

TRAIN_DIR = COMBINED_DIR / 'train'
VAL_DIR = COMBINED_DIR / 'val'
TEST_DIR = COMBINED_DIR / 'test'

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=val_transform)

# Authoritative class names from dataset
CLASS_NAMES = train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

# Check class alignment with API expectations
API_CLASSES = ['acne', 'eczema', 'fungal', 'healthy', 'psoriasis', 'scabies']
if list(CLASS_NAMES) != API_CLASSES:
    missing = set(API_CLASSES) - set(CLASS_NAMES)
    extra = set(CLASS_NAMES) - set(API_CLASSES)
    print(f"NOTE: Training with {NUM_CLASSES} classes (API expects {len(API_CLASSES)}).")
    if missing:
        print(f"  Missing from training data: {sorted(missing)} — no data available for these.")
    if extra:
        print(f"  Extra classes not in API: {sorted(extra)}")
    print("  The exported config will reflect the actual trained classes.")

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Mapping: {train_dataset.class_to_idx}")

In [ ]:
# Weighted sampling to handle class imbalance
targets = train_dataset.targets
class_counts = Counter(targets)
total = len(targets)
class_weights = {c: total / count for c, count in class_counts.items()}
sample_weights = [class_weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
print("Data loaders ready")

## Step 7: Model (DINOv2 ViT-B/14)

Uses DINOv2 pre-trained on skin disease images (`Jayanth2002/dinov2-base-finetuned-SkinDisease`).
Backbone is frozen — only the classification head is trained (transfer learning).

In [ ]:
NUM_EPOCHS = 10

class SkinDiseaseClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.dinov2 = AutoModel.from_pretrained("Jayanth2002/dinov2-base-finetuned-SkinDisease")

        # Freeze entire backbone (transfer learning)
        for param in self.dinov2.parameters():
            param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(768, num_classes)
        )

    def forward(self, pixel_values):
        outputs = self.dinov2(pixel_values)
        cls_token = outputs.last_hidden_state[:, 0]  # [CLS] token
        return self.classifier(cls_token)

model = SkinDiseaseClassifier(NUM_CLASSES).to(device)

optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss(label_smoothing=0.15)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"DINOv2 ViT-B/14 (skin-disease pretrained) → {NUM_CLASSES} classes: {CLASS_NAMES}")
print(f"Trainable: {n_trainable:,} / {n_total:,} params ({n_trainable/n_total*100:.1f}%)")

## Step 8: Training

Frozen backbone transfer learning — only the classification head is trained.
DINOv2 features are already excellent for skin disease classification, so a simple
linear head with dropout achieves strong results in just 10 epochs.

In [ ]:
# ── MixUp helpers ─────────────────────────────────────────────────────────────
def mixup_data(x, y, alpha=0.2):
    """Apply MixUp augmentation at the batch level."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Compute MixUp loss as weighted combination."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

MIXUP_PROB = 0.5

# Mixed precision scaler (disabled on CPU)
use_amp = device.type == 'cuda'
scaler = torch.amp.GradScaler(enabled=use_amp)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, correct, total, n_batches = 0, 0, 0, 0
    flushed = True
    optimizer.zero_grad()
    pbar = tqdm(loader, desc="Train")
    for batch_idx, (images, labels) in enumerate(pbar):
        images, labels = images.to(device), labels.to(device)

        use_mixup = random.random() < MIXUP_PROB
        if use_mixup:
            images, targets_a, targets_b, lam = mixup_data(images, labels)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(images)
            if use_mixup:
                loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
            else:
                loss = criterion(outputs, labels)
            loss_scaled = loss / ACCUM_STEPS

        scaler.scale(loss_scaled).backward()
        flushed = False

        if (batch_idx + 1) % ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            flushed = True

        loss_sum += loss.item()
        n_batches += 1
        _, preds = outputs.max(1)
        total += labels.size(0)
        if not use_mixup:
            correct += preds.eq(labels).sum().item()
        else:
            # Count accuracy against dominant class only
            correct += preds.eq(targets_a).sum().item() if lam > 0.5 else preds.eq(targets_b).sum().item()
        pbar.set_postfix(loss=f"{loss_sum/n_batches:.4f}", acc=f"{100*correct/total:.1f}%")

    # Flush remaining gradients
    if not flushed:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    return loss_sum / n_batches, correct / total

def validate(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total, n_batches = 0, 0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Val"):
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)
            loss_sum += loss.item()
            n_batches += 1
            _, preds = outputs.max(1)
            total += labels.size(0)
            correct += preds.eq(labels).sum().item()
    return loss_sum / n_batches, correct / total

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
PATIENCE = 5
BEST_MODEL_PATH = f"{MODEL_PATH}/best_model.pth"

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0
patience_counter = 0

print(f"{'='*60}")
print(f"Training: {len(train_dataset)} images, {NUM_EPOCHS} epochs, patience={PATIENCE}")
print(f"Batch: {BATCH_SIZE} | Accum: {ACCUM_STEPS} | Effective: {BATCH_SIZE * ACCUM_STEPS}")
print(f"Mixed precision: {'enabled' if use_amp else 'disabled (CPU)'}")
print(f"{'='*60}")

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} (lr={scheduler.get_last_lr()[0]:.2e})")

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"  Train: loss={train_loss:.4f} acc={train_acc*100:.1f}%")
    print(f"  Val:   loss={val_loss:.4f} acc={val_acc*100:.1f}%")
    print(f"  Gap:   {(train_acc - val_acc)*100:.1f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        # Also cache to Drive for persistence across runtime resets
        shutil.copy2(BEST_MODEL_PATH, f"{DRIVE_DATA}/best_model.pth")
        print(f"  >> Saved best model ({val_acc*100:.1f}%) — local + Drive")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("  >> Early stopping")
            break

print(f"\nBest Val Accuracy: {best_val_acc*100:.1f}%")

In [ ]:
# ── Plot training history ─────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot([a*100 for a in history['train_acc']], label='Train')
ax2.plot([a*100 for a in history['val_acc']], label='Val')
ax2.set_title('Accuracy (%)')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/training_history.png", dpi=150)
plt.show()

## Step 9: Evaluation

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────────────
model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

test_acc = np.mean(np.array(all_preds) == np.array(all_labels))
print(f"\nTest Accuracy: {test_acc*100:.2f}%")
print("\nClassification Report:")
print("=" * 50)
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=3))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/confusion_matrix.png", dpi=150)
plt.show()

## Step 10: Grad-CAM & Attention Visualization

GradCAM works with DINOv2 by temporarily enabling gradients on the frozen backbone.
We also show self-attention maps from the last transformer block for comparison.

In [ ]:
import cv2

# ── GradCAM for DINOv2 (ViT) ─────────────────────────────────────────────────
class GradCAM:
    """GradCAM adapted for Vision Transformers.
    Hooks into a ViT encoder layer and reshapes patch tokens to a spatial grid."""

    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        self._fwd = target_layer.register_forward_hook(self._save_act)
        self._bwd = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, m, i, o):
        out = o[0] if isinstance(o, tuple) else o
        self.activations = out.detach()

    def _save_grad(self, m, gi, go):
        out = go[0] if isinstance(go, tuple) else go
        self.gradients = out.detach()

    def generate(self, x, target=None):
        self.model.eval()
        # Temporarily enable gradients on backbone for GradCAM
        for param in self.model.dinov2.parameters():
            param.requires_grad = True

        with torch.enable_grad():
            out = self.model(x)
            if target is None:
                target = out.argmax(1).item()
            self.model.zero_grad()
            one_hot = torch.zeros_like(out)
            one_hot[0, target] = 1
            out.backward(gradient=one_hot)

        # Re-freeze backbone
        for param in self.model.dinov2.parameters():
            param.requires_grad = False

        # activations/gradients shape: (batch, num_tokens, hidden_dim)
        # Skip CLS token (index 0), use only patch tokens
        act = self.activations[0, 1:]   # (num_patches, hidden_dim)
        grad = self.gradients[0, 1:]    # (num_patches, hidden_dim)

        # Standard GradCAM: weights are per-channel (hidden_dim), averaged across patches
        weights = grad.mean(dim=0)      # (hidden_dim,)
        cam = torch.relu((weights * act).sum(dim=-1))  # (num_patches,)

        # Reshape to spatial grid (224/14 = 16 patches per side)
        num_patches = int(cam.shape[0] ** 0.5)
        assert num_patches * num_patches == cam.shape[0], \
            f"Non-square patch grid: {cam.shape[0]} patches"
        cam = cam.reshape(num_patches, num_patches).cpu().numpy()

        # Normalize
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

    def remove_hooks(self):
        self._fwd.remove()
        self._bwd.remove()

# ── Attention map extraction ──────────────────────────────────────────────────
def get_dinov2_attention(model, img_tensor):
    """Extract attention map from DINOv2's last transformer block."""
    model.eval()
    with torch.no_grad():
        outputs = model.dinov2(img_tensor, output_attentions=True)

    if not outputs.attentions:
        print("WARNING: No attention weights returned")
        return np.ones((16, 16)) * 0.5

    attn = outputs.attentions[-1]  # (batch, heads, tokens, tokens)
    attn_map = attn[0].mean(dim=0)[0, 1:]  # CLS→patches, averaged over heads
    num_patches = int(attn_map.shape[0] ** 0.5)
    attn_map = attn_map.reshape(num_patches, num_patches).cpu().numpy()
    attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
    return attn_map

def apply_heatmap(img, cam, alpha=0.5):
    """Overlay heatmap on image."""
    cam_resized = cv2.resize(cam, (img.shape[1], img.shape[0]))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    return (heatmap * alpha + img * (1 - alpha)).astype(np.uint8)

In [ ]:
# ── GradCAM + Attention for each class ────────────────────────────────────────
gradcam = GradCAM(model, model.dinov2.encoder.layer[-1])

fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(14, NUM_CLASSES * 3))

for idx, cls in enumerate(CLASS_NAMES):
    cls_path = TEST_DIR / cls
    images = list(cls_path.glob("*"))
    if not images:
        for col in range(4):
            axes[idx, col].text(0.5, 0.5, f'No test images\nfor {cls}',
                                ha='center', va='center', fontsize=12)
            axes[idx, col].axis('off')
        continue

    img = Image.open(images[0]).convert('RGB')
    img_np = np.array(img.resize((IMG_SIZE, IMG_SIZE)))
    input_tensor = val_transform(img).unsqueeze(0).to(device)

    # GradCAM
    cam = gradcam.generate(input_tensor)
    gradcam_overlay = apply_heatmap(img_np, cam)

    # Attention map
    attn_map = get_dinov2_attention(model, input_tensor)
    attn_overlay = apply_heatmap(img_np, attn_map)

    # Prediction
    with torch.no_grad():
        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            out = model(input_tensor)
        prob = torch.softmax(out, dim=1)
        pred_idx = out.argmax(1).item()
        conf = prob[0, pred_idx].item()

    axes[idx, 0].imshow(img_np); axes[idx, 0].set_title(f'Original: {cls}'); axes[idx, 0].axis('off')
    axes[idx, 1].imshow(gradcam_overlay); axes[idx, 1].set_title('Grad-CAM'); axes[idx, 1].axis('off')
    axes[idx, 2].imshow(attn_overlay); axes[idx, 2].set_title('Attention'); axes[idx, 2].axis('off')
    axes[idx, 3].imshow(img_np); axes[idx, 3].set_title(f'Pred: {CLASS_NAMES[pred_idx]} ({conf*100:.0f}%)'); axes[idx, 3].axis('off')

plt.tight_layout()
plt.savefig(f"{MODEL_PATH}/gradcam_attention.png", dpi=150)
plt.show()
gradcam.remove_hooks()

## Step 11: Export for Deployment

Saves model + config locally, then copies the bundle to Drive (already mounted from Step 1).

In [ ]:
# ── Save model + config locally ────────────────────────────────────────────────
torch.save(model.state_dict(), f"{MODEL_PATH}/skin_disease_model.pth")

config = {
    'model_name': 'skin_disease',
    'architecture': 'dinov2_vitb14_skin',
    'backbone': 'Jayanth2002/dinov2-base-finetuned-SkinDisease',
    'num_classes': NUM_CLASSES,
    'class_names': list(CLASS_NAMES),
    'input_size': IMG_SIZE,
    'hidden_dim': 768,
    'normalize': {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]},
    'test_accuracy': float(test_acc),
    'best_val_accuracy': float(best_val_acc)
}

with open(f"{MODEL_PATH}/skin_disease_config.json", 'w') as f:
    json.dump(config, f, indent=2)

print("Saved locally:")
for f in os.listdir(MODEL_PATH):
    size = os.path.getsize(f"{MODEL_PATH}/{f}") / 1e6
    print(f"  {f} ({size:.1f} MB)")
print(f"\nConfig: {json.dumps(config, indent=2)}")

In [ ]:
# ── Export: zip + save to Drive + download ────────────────────────────────────
zip_path = f"{MODEL_PATH}/skin_disease_deployment.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(MODEL_PATH):
        if fname.endswith(('.pth', '.json', '.png')):
            zf.write(f"{MODEL_PATH}/{fname}", fname)

print(f"Bundle: {os.path.getsize(zip_path)/1e6:.1f} MB")

# Drive already mounted — copy bundle there for persistence
DRIVE_EXPORT = "/content/drive/MyDrive/skin_disease_project/models"
os.makedirs(DRIVE_EXPORT, exist_ok=True)
shutil.copy2(zip_path, f"{DRIVE_EXPORT}/skin_disease_deployment.zip")
print(f"Copied to Drive: {DRIVE_EXPORT}/skin_disease_deployment.zip")

# Also offer direct download
from google.colab import files
files.download(zip_path)
print("\nDone! Extract the zip and place files in api/weights/")